# GPU & Environment Diagnostic Tool
Run this notebook to verify that your environment is correctly set up for the Deepfake Defense project.

In [1]:
import torch
import sys
import platform

print("--- System Info ---")
print(f"Python Version: {sys.version.split()[0]}")
print(f"OS: {platform.system()} {platform.release()}")
print(f"PyTorch Version: {torch.__version__}")

--- System Info ---
Python Version: 3.10.11
OS: Windows 10
PyTorch Version: 2.5.1+cu121


## Step 1: Check CUDA Availability
If this returns `False`, your PyTorch is not linked to your GPU.

In [2]:
cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")

if cuda_available:
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"CUDNN Version: {torch.backends.cudnn.version()}")
else:
    print("❌ ERROR: CUDA is not available. Please reinstall PyTorch with the correct CUDA version.")

CUDA Available: True
CUDA Version: 12.1
CUDNN Version: 90100


## Step 2: GPU Hardware Details
This checks your VRAM to determine if you need 'Low VRAM' mode.

In [3]:
if cuda_available:
    device_count = torch.cuda.device_count()
    print(f"GPUs Detected: {device_count}")
    
    for i in range(device_count):
        gpu_name = torch.cuda.get_device_name(i)
        gpu_props = torch.cuda.get_device_properties(i)
        total_vram_gb = gpu_props.total_memory / 1e9
        
        print(f"\n--- GPU {i}: {gpu_name} ---")
        print(f"Total VRAM: {total_vram_gb:.2f} GB")
        print(f"Compute Capability: {gpu_props.major}.{gpu_props.minor}")
        
        # Traffic Light Assessment
        if total_vram_gb >= 8.0:
            print("✅ STATUS: GREEN (Good to go!)")
        elif total_vram_gb >= 4.0:
            print("⚠️ STATUS: YELLOW (Tight fit. Ensure BATCH_SIZE=1 and 'enable_model_cpu_offload()' is used.)")
        else:
            print("❌ STATUS: RED (VRAM is too low for Stable Diffusion. You may face OutOfMemory errors.)")
else:
    print("No GPU detected to analyze.")

GPUs Detected: 1

--- GPU 0: NVIDIA GeForce RTX 3050 Laptop GPU ---
Total VRAM: 4.29 GB
Compute Capability: 8.6
⚠️ STATUS: YELLOW (Tight fit. Ensure BATCH_SIZE=1 and 'enable_model_cpu_offload()' is used.)


## Step 3: Functional Test
We will now move a tensor to the GPU and perform a matrix multiplication to prove it works.

In [5]:
if cuda_available:
    try:
        # Create random tensors
        x = torch.rand(1000, 1000).to("cuda")
        y = torch.rand(1000, 1000).to("cuda")
        
        # Perform multiplication
        z = torch.matmul(x, y)
        
        print(f"Calculation Result Shape: {z.shape}")
        print(f"Result Device: {z.device}")
        print("✅ SUCCESS: GPU is functioning correctly for math operations.")
    except Exception as e:
        print(f"❌ FAILURE: An error occurred during GPU calculation.\nError: {e}")
else:
    print("Skipping functional test (No GPU).")

Calculation Result Shape: torch.Size([1000, 1000])
Result Device: cuda:0
✅ SUCCESS: GPU is functioning correctly for math operations.
